# ODI to Databricks Migration

**Source File:** `TARGET_ODI_SQL.sql`

**Conversion Timestamp:** 2024-07-30 12:00:00 UTC

**Description:** This notebook converts a simple ODI INSERT statement from Oracle HR schema to a Databricks Spark SQL equivalent, including schema and table name standardization.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "1. ETL Job Type");
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "2. Datasource Num ID");
dbutils.widgets.text("ETL_PROC_WID", "-1", "3. ETL Process ID");
dbutils.widgets.text("ODI_SESS_NO", "-1", "4. ODI Session Number");
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "1900-01-01 00:00:00", "5. ETL Current Extract Time");
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "6. ETL Last Extract Time");

# ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS SELECT '${ETL_JOB_TYPE}' AS etl_job_type;
CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS SELECT ${DATASOURCE_NUM_ID} AS datasource_num_id;
CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS SELECT ${ETL_PROC_WID} AS etl_proc_wid;
CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS SELECT '${ODI_SESS_NO}' AS odi_sess_no;
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS SELECT to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS SELECT to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time;

In [ ]:
display(spark.sql("""
  SELECT 
    (SELECT etl_job_type FROM v_etl_job_type) AS ETL_JOB_TYPE,
    (SELECT datasource_num_id FROM v_datasource_num_id) AS DATASOURCE_NUM_ID,
    (SELECT etl_proc_wid FROM v_etl_proc_wid) AS ETL_PROC_WID,
    (SELECT odi_sess_no FROM v_odi_sess_no) AS ODI_SESS_NO,
    (SELECT etl_current_extract_time FROM v_etl_current_extract_time) AS ETL_CURRENT_EXTRACT_TIME,
    (SELECT etl_last_extract_time FROM v_etl_last_extract_time) AS ETL_LAST_EXTRACT_TIME
"""))

# Load Target

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Insert into target HR.TRG_EMP
INSERT 
  INTO workspace.hr.trg_emp
  (
    employee_id ,
    first_name ,
    last_name ,
    email ,
    phone_number ,
    hire_date ,
    job_id ,
    salary ,
    commission_pct ,
    manager_id ,
    department_id 
  ) 
SELECT 
  employees.employee_id ,
  employees.first_name ,
  employees.last_name ,
  employees.email ,
  employees.phone_number ,
  employees.hire_date ,
  employees.job_id ,
  employees.salary ,
  employees.commission_pct ,
  employees.manager_id ,
  employees.department_id  
FROM 
  workspace.hr.employees AS employees;

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_emp FROM workspace.hr.trg_emp;

# Conversion Notes

1.  **Schema and Table Naming:** Original Oracle schema `HR` converted to `workspace.hr`. Table names `TRG_EMP` and `EMPLOYEES` converted to lowercase `trg_emp` and `employees` respectively.
2.  **Oracle Hints Removal:** The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable in Databricks Spark SQL.
3.  **SCEN_TASK_NO:** Empty `SCEN_TASK_NO` blocks {10}, {20} were omitted. `SCEN_TASK_NO {30}` is noted as a comment above the relevant SQL statement.
4.  **Parameter Handling:** While the original SQL did not contain ODI parameters, a standard set of `dbutils.widgets` and corresponding temporary views (`v_etl_job_type`, `v_datasource_num_id`, etc.) have been included as per the standard notebook template for Databricks migrations.